# Experiment 1: ViT-L/16 Baseline & Augmented

## Architecture

Standard Vision Transformer (Dosovitskiy et al., 2021) with no equivariance guarantee.

An image $\mathbf{I} \in \mathbb{R}^{224 \times 224 \times 3}$ is split into $14 \times 14 = 196$ non-overlapping patches of size $16 \times 16$, linearly projected to $\mathbb{R}^{1024}$. The `[CLS]` token after 24 transformer blocks is taken as the embedding $z \in \mathbb{R}^{1024}$.

## Siamese Framework

$$f_\theta(x_1, x_2) = \sigma\!\bigl(h_\psi(\varphi_\theta(x_1),\, \varphi_\theta(x_2))\bigr)$$

where $\varphi_\theta$ is the shared encoder, $h_\psi$ is the fusion module:

$$h_\psi(z_1, z_2) = W_2\,\mathrm{ReLU}\!\bigl(W_1\,[|z_1 - z_2|\;\|\;z_1 \odot z_2] + b_1\bigr) + b_2$$

## Loss Function

$$\mathcal{L} = \mathcal{L}_{\mathrm{BCE}} + \lambda\,\mathcal{L}_{\mathrm{contr}}^{L^2}$$

$$\mathcal{L}_{\mathrm{contr}}^{L^2}(z_1, z_2, y) = y\,\|z_1 - z_2\|_2^2 + (1 - y)\,\max\!\bigl(0,\, m - \|z_1 - z_2\|_2\bigr)^2$$

## Runs in this notebook

| Run | Augmentation | $\lambda$ | Purpose |
|-----|-------------|-----------|----------|
| `vit_baseline_no_reg` | Block P only | 0.0 | Pure baseline |
| `vit_baseline_reg` | Block P only | 0.1 | L2 regularizer effect |
| `vit_aug_no_reg` | Block P + G | 0.0 | Geometric augmentation effect |
| `vit_aug_reg` | Block P + G | 0.1 | Best non-equivariant config |

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import torch
import yaml
import logging
import matplotlib.pyplot as plt
import numpy as np

from src.utils.seed import set_seed
from src.data_module.augmentations import build_train_transform, build_val_transform
from src.data_module.coco_dataset import build_coco_dataloaders
from src.data_module.domainnet_dataset import DomainNetEvalDataset
from src.encoders import EncoderFactory
from src.losses.composite import CompositeLoss
from src.models.siamese import SiameseNet
from src.training.trainer import Trainer
from src.validation.evaluator import DomainNetEvaluator
from src.validation.tsne import compute_tsne_embeddings, plot_tsne
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Load config
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

set_seed(cfg['seed'])

# Build data (baseline: Block P on train, clean val - see default.yaml)
train_tfm = build_train_transform(cfg)
val_tfm = build_val_transform(cfg)
dataloaders = build_coco_dataloaders(cfg, train_tfm, val_tfm)
print(f'Train batches: {len(dataloaders["train"])}, Val batches: {len(dataloaders["val"])}')

## Run 1: ViT-L/16 Baseline (Block P, no regularizer)

In [ ]:
# --- Run: vit_baseline_no_reg ---
set_seed(cfg['seed'])
encoder = EncoderFactory('vit_baseline', freeze=True)
model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)

criterion = CompositeLoss(
    bce_weight_pos=0.3, bce_weight_neg=0.7,
    contrastive_margin=1.0, lambda_reg=0.0,
)
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)

trainer = Trainer(
    model=model, criterion=criterion, optimizer=optimizer,
    scheduler=scheduler, device=device,
    checkpoint_dir='../outputs/checkpoints/vit_baseline_no_reg',
)
history_bl_noreg = trainer.fit(
    dataloaders['train'], dataloaders['val'],
    num_epochs=cfg['training']['num_epochs'],
    run_name='vit_baseline_no_reg',
)

## Run 2: ViT-L/16 Baseline (Block P, with L2 regularizer)

In [ ]:
# --- Run: vit_baseline_reg ---
set_seed(cfg['seed'])
encoder = EncoderFactory('vit_baseline', freeze=True)
model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)

criterion = CompositeLoss(
    bce_weight_pos=0.3, bce_weight_neg=0.7,
    contrastive_margin=1.0, lambda_reg=0.1,
)
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)

trainer = Trainer(
    model=model, criterion=criterion, optimizer=optimizer,
    scheduler=scheduler, device=device,
    checkpoint_dir='../outputs/checkpoints/vit_baseline_reg',
)
history_bl_reg = trainer.fit(
    dataloaders['train'], dataloaders['val'],
    num_epochs=cfg['training']['num_epochs'],
    run_name='vit_baseline_reg',
)

## Run 3-4: ViT-L/16 Augmented (Block P + G)

**Block G** adds: `RandomHorizontalFlip(p=0.5)`, `RandomVerticalFlip(p=0.5)`, `RandomChoice({0, 90, 180, 270})`, `RandomResizedCrop(scale=[0.7,1.0])`

In [ ]:
# Block G on train only; val transform unchanged (clean by default)
_prev_skip_geo = cfg['augmentation']['skip_geometric']
cfg['augmentation']['skip_geometric'] = False
aug_train_tfm = build_train_transform(cfg)
aug_val_tfm = build_val_transform(cfg)
aug_dataloaders = build_coco_dataloaders(cfg, aug_train_tfm, aug_val_tfm)
cfg['augmentation']['skip_geometric'] = _prev_skip_geo
print(f'Augmented - train batches: {len(aug_dataloaders["train"])}, val batches: {len(aug_dataloaders["val"])}')

In [ ]:
for run_name, lam in [('vit_aug_no_reg', 0.0), ('vit_aug_reg', 0.1)]:
    print(f'\n{"="*60}')
    print(f'Run: {run_name}, lambda={lam}')
    print(f'{"="*60}')
    
    set_seed(cfg['seed'])
    encoder = EncoderFactory('vit_augmented', freeze=True)
    model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)
    
    criterion = CompositeLoss(
        bce_weight_pos=0.3, bce_weight_neg=0.7,
        contrastive_margin=1.0, lambda_reg=lam,
    )
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)
    
    trainer = Trainer(
        model=model, criterion=criterion, optimizer=optimizer,
        scheduler=scheduler, device=device,
        checkpoint_dir=f'../outputs/checkpoints/{run_name}',
    )
    trainer.fit(
        aug_dataloaders['train'], aug_dataloaders['val'],
        num_epochs=cfg['training']['num_epochs'],
        run_name=run_name,
    )

## DomainNet Evaluation

In [ ]:
domainnet_ds = DomainNetEvalDataset(
    root=cfg['data']['domainnet_root'],
    domains=cfg['data']['domainnet_domains'],
    pairs_per_domain=cfg['data']['pairs_per_domain'],
    image_size=cfg['data']['image_size'],
)

evaluator = DomainNetEvaluator(
    model=model, dataset=domainnet_ds, device=device,
)
report = evaluator.run()
print(json.dumps(report, indent=2))

## t-SNE Visualization

t-SNE projects the 1024-dimensional embeddings to 2D for visual inspection of domain separation.

In [ ]:
tsne_loader = DataLoader(domainnet_ds, batch_size=32, shuffle=False, num_workers=2)
coords, domains = compute_tsne_embeddings(
    model, tsne_loader, device=device, perplexity=30,
)
fig = plot_tsne(
    coords, domains,
    title='t-SNE: ViT-L/16 (last trained variant)',
    save_path='../outputs/figures/vit_baseline_tsne.pdf',
)
plt.show()

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, label in [(history_bl_noreg, 'baseline no_reg'), (history_bl_reg, 'baseline reg')]:
    epochs = [h['epoch'] for h in hist]
    train_loss = [h['train']['loss'] for h in hist]
    val_loss = [h['val']['loss'] for h in hist]
    axes[0].plot(epochs, train_loss, label=f'{label} (train)')
    axes[1].plot(epochs, val_loss, label=f'{label} (val)')

axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')
fig.tight_layout()
fig.savefig('../outputs/figures/vit_baseline_curves.pdf', dpi=300)
plt.show()